# 04 — Pipeline Validation

Validación integral del pipeline modular de Machine Learning.

## Objetivos

- Verificar que el pipeline guardado puede cargarse y utilizarse.
- Comprobar consistencia entre entrenamiento y predicción.
- Validar esquemas de entrada.
- Evaluar reproducibilidad.
- Comparar resultados del notebook con los módulos de `src/`.
- Detectar errores comunes antes del despliegue.


## 1. Configuración del entorno

Este notebook debe ejecutarse desde la raíz del repositorio.

En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 2. Importaciones


In [ ]:
import hashlib
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from src.data import load_customer_data
from src.features import (
    build_preprocessing_pipeline,
    split_features_target,
)
from src.models import (
    load_model,
    predict_customers,
    train_model,
)
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.pipeline_validation")


## 3. Rutas y artefactos esperados


In [ ]:
DATA_PATH = get_project_path(
    "data",
    "raw",
    "customer_churn.csv",
)

MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "selected_churn_pipeline.joblib",
)

METRICS_PATH = get_project_path(
    "reports",
    "metrics",
    "model_comparison.json",
)

required_paths = {
    "data": DATA_PATH,
    "model": MODEL_PATH,
    "metrics": METRICS_PATH,
}

pd.DataFrame({
    "artifact": required_paths.keys(),
    "path": [str(path) for path in required_paths.values()],
    "exists": [path.exists() for path in required_paths.values()],
})


## 4. Carga y validación del dataset


In [ ]:
df = load_customer_data(
    DATA_PATH,
    validate=True,
)

print(df.shape)
df.head()


In [ ]:
data_profile = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_customer_ids": int(
        df["customer_id"].duplicated().sum()
    ),
    "churn_rate": float(df["churn"].mean()),
}

data_profile


## 5. Firma reproducible del dataset


In [ ]:
def dataframe_signature(dataframe: pd.DataFrame) -> str:
    normalized = (
        dataframe
        .sort_values("customer_id")
        .reset_index(drop=True)
        .to_csv(index=False)
        .encode("utf-8")
    )

    return hashlib.sha256(normalized).hexdigest()

dataset_signature = dataframe_signature(df)
dataset_signature


La firma SHA-256 permite detectar cambios en el dataset.  
Si cambia un valor, una fila o el orden lógico normalizado, la firma será distinta.


## 6. Carga del pipeline entrenado


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el pipeline entrenado. "
        "Ejecute primero 02_model_development.ipynb."
    )

saved_pipeline = load_model(MODEL_PATH)

print(type(saved_pipeline))
print(saved_pipeline.named_steps.keys())


## 7. Validación de pasos del pipeline


In [ ]:
expected_steps = [
    "preprocessor",
    "classifier",
]

actual_steps = list(
    saved_pipeline.named_steps.keys()
)

pipeline_structure_check = {
    "expected_steps": expected_steps,
    "actual_steps": actual_steps,
    "matches": expected_steps == actual_steps,
}

pipeline_structure_check


## 8. Validación de transformación


In [ ]:
X, y = split_features_target(df)

preprocessor = saved_pipeline.named_steps[
    "preprocessor"
]

transformed_sample = preprocessor.transform(
    X.head(10)
)

print(f"Input shape: {X.head(10).shape}")
print(f"Transformed shape: {transformed_sample.shape}")
print(f"Contains NaN: {np.isnan(transformed_sample).any()}")


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(f"Generated features: {len(feature_names)}")
feature_names[:20]


## 9. Consistencia entre `predict()` y umbral 0.50


In [ ]:
sample = df.head(50).copy()

direct_predictions = saved_pipeline.predict(
    X.head(50)
)

probabilities = saved_pipeline.predict_proba(
    X.head(50)
)[:, 1]

threshold_predictions = (
    probabilities >= 0.50
).astype(int)

prediction_consistency = np.array_equal(
    direct_predictions,
    threshold_predictions,
)

prediction_consistency


## 10. Validación del módulo `predict_customers`


In [ ]:
module_predictions = predict_customers(
    saved_pipeline,
    sample,
    threshold=0.50,
)

module_predictions.head()


In [ ]:
module_consistency = np.array_equal(
    module_predictions[
        "churn_prediction"
    ].to_numpy(),
    direct_predictions,
)

probability_consistency = np.allclose(
    module_predictions[
        "churn_probability"
    ].to_numpy(),
    probabilities,
)

{
    "prediction_consistency": module_consistency,
    "probability_consistency": probability_consistency,
}


## 11. Reproducibilidad del entrenamiento


In [ ]:
result_run_1 = train_model(
    df,
    test_size=0.20,
    random_state=42,
)

result_run_2 = train_model(
    df,
    test_size=0.20,
    random_state=42,
)

reproducibility_metrics = pd.DataFrame({
    "run_1": result_run_1.metrics,
    "run_2": result_run_2.metrics,
})

reproducibility_metrics["difference"] = (
    reproducibility_metrics["run_1"]
    - reproducibility_metrics["run_2"]
).abs()

reproducibility_metrics


In [ ]:
metrics_reproducible = (
    reproducibility_metrics[
        "difference"
    ].max()
    < 1e-12
)

metrics_reproducible


## 12. Comparación del modelo guardado con un reentrenamiento


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

saved_probabilities = saved_pipeline.predict_proba(
    X_test
)[:, 1]

retrained_probabilities = (
    result_run_1.pipeline.predict_proba(
        X_test
    )[:, 1]
)

probability_difference = np.abs(
    saved_probabilities
    - retrained_probabilities
)

pd.Series(
    probability_difference
).describe()


La diferencia debe ser cero o muy pequeña si:

- se usó el mismo dataset;
- se usó la misma semilla;
- se usaron los mismos hiperparámetros;
- no cambiaron las dependencias relevantes.


## 13. Validación de inferencia sin target


In [ ]:
inference_df = df.drop(
    columns=["churn"]
).head(20)

inference_predictions = predict_customers(
    saved_pipeline,
    inference_df,
)

inference_predictions.head()


In [ ]:
assert len(inference_predictions) == len(
    inference_df
)
assert inference_predictions[
    "churn_probability"
].between(0, 1).all()
assert set(
    inference_predictions[
        "churn_prediction"
    ].unique()
).issubset({0, 1})

print("Inference dataset validation passed.")


## 14. Prueba con categoría desconocida


In [ ]:
unknown_category_df = inference_df.head(1).copy()

unknown_category_df.loc[
    unknown_category_df.index[0],
    "region",
] = "Región Nueva"

unknown_category_df.loc[
    unknown_category_df.index[0],
    "payment_method",
] = "Billetera digital"

unknown_category_prediction = predict_customers(
    saved_pipeline,
    unknown_category_df,
)

unknown_category_prediction


El pipeline debe aceptar categorías desconocidas debido a:

```python
OneHotEncoder(handle_unknown="ignore")
```


## 15. Prueba de columnas faltantes


In [ ]:
broken_df = inference_df.drop(
    columns=["monthly_fee"]
)

try:
    predict_customers(
        saved_pipeline,
        broken_df,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)


## 16. Prueba de umbral inválido


In [ ]:
for invalid_threshold in [
    0.0,
    1.0,
    -0.2,
    1.5,
]:
    try:
        predict_customers(
            saved_pipeline,
            inference_df.head(1),
            threshold=invalid_threshold,
        )
    except ValueError as error:
        print(
            invalid_threshold,
            "->",
            error,
        )


## 17. Prueba de estabilidad por lote


In [ ]:
single_batch = predict_customers(
    saved_pipeline,
    inference_df,
)

split_batch = pd.concat(
    [
        predict_customers(
            saved_pipeline,
            inference_df.iloc[:10],
        ),
        predict_customers(
            saved_pipeline,
            inference_df.iloc[10:],
        ),
    ]
).sort_index()

batch_predictions_equal = np.array_equal(
    single_batch[
        "churn_prediction"
    ].to_numpy(),
    split_batch[
        "churn_prediction"
    ].to_numpy(),
)

batch_probabilities_equal = np.allclose(
    single_batch[
        "churn_probability"
    ].to_numpy(),
    split_batch[
        "churn_probability"
    ].to_numpy(),
)

{
    "predictions_equal": batch_predictions_equal,
    "probabilities_equal": batch_probabilities_equal,
}


## 18. Validación de serialización


In [ ]:
TEMP_MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "validation_roundtrip.joblib",
    create_parent=True,
)

joblib.dump(
    saved_pipeline,
    TEMP_MODEL_PATH,
)

roundtrip_pipeline = joblib.load(
    TEMP_MODEL_PATH
)

roundtrip_predictions = (
    roundtrip_pipeline.predict_proba(
        X.head(25)
    )[:, 1]
)

original_predictions = (
    saved_pipeline.predict_proba(
        X.head(25)
    )[:, 1]
)

roundtrip_consistency = np.allclose(
    roundtrip_predictions,
    original_predictions,
)

roundtrip_consistency


## 19. Ejecución de pruebas automatizadas


In [ ]:
import subprocess

test_process = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(test_process.stdout)

if test_process.returncode != 0:
    print(test_process.stderr)


## 20. Resumen de validaciones


In [ ]:
validation_summary = {
    "dataset_signature": dataset_signature,
    "pipeline_structure_valid": pipeline_structure_check[
        "matches"
    ],
    "transformed_data_has_no_nan": bool(
        not np.isnan(
            transformed_sample
        ).any()
    ),
    "predict_matches_threshold": bool(
        prediction_consistency
    ),
    "module_predictions_match": bool(
        module_consistency
    ),
    "module_probabilities_match": bool(
        probability_consistency
    ),
    "training_metrics_reproducible": bool(
        metrics_reproducible
    ),
    "batch_predictions_stable": bool(
        batch_predictions_equal
    ),
    "batch_probabilities_stable": bool(
        batch_probabilities_equal
    ),
    "serialization_roundtrip_valid": bool(
        roundtrip_consistency
    ),
    "pytest_return_code": int(
        test_process.returncode
    ),
}

validation_summary


## 21. Exportación del reporte


In [ ]:
VALIDATION_REPORT_PATH = get_project_path(
    "reports",
    "metrics",
    "pipeline_validation.json",
    create_parent=True,
)

VALIDATION_REPORT_PATH.write_text(
    json.dumps(
        validation_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(VALIDATION_REPORT_PATH)


## 22. Criterio de aprobación


In [ ]:
boolean_checks = [
    value
    for key, value in validation_summary.items()
    if key != "dataset_signature"
    and key != "pytest_return_code"
]

pipeline_approved = (
    all(boolean_checks)
    and validation_summary[
        "pytest_return_code"
    ] == 0
)

print(
    "PIPELINE APPROVED"
    if pipeline_approved
    else "PIPELINE REQUIRES REVIEW"
)


## 23. Preguntas para estudiantes

1. ¿Por qué se debe guardar el preprocesamiento junto con el modelo?
2. ¿Qué factores pueden romper la reproducibilidad?
3. ¿Por qué se compara inferencia individual y batch?
4. ¿Qué pasaría si `OneHotEncoder` no permitiera categorías desconocidas?
5. ¿Qué diferencia existe entre validar datos y validar el pipeline?
6. ¿Qué pruebas deberían ejecutarse antes de desplegar una nueva versión?


## 24. Checklist de cierre

- [ ] Se verificó la existencia de artefactos.
- [ ] Se generó la firma del dataset.
- [ ] Se validaron los pasos del pipeline.
- [ ] Se verificó ausencia de NaN después de transformar.
- [ ] Se comparó `predict()` con el umbral 0.50.
- [ ] Se validó el módulo de predicción.
- [ ] Se comprobó reproducibilidad.
- [ ] Se probaron categorías desconocidas.
- [ ] Se probaron columnas faltantes.
- [ ] Se validó inferencia batch.
- [ ] Se comprobó serialización.
- [ ] Se ejecutaron pruebas automatizadas.
- [ ] Se exportó el reporte.


## Resultado esperado

Este notebook funciona como una puerta de calidad antes del despliegue.

El pipeline se considera aprobado cuando:

- transforma correctamente;
- predice consistentemente;
- produce resultados reproducibles;
- acepta entradas válidas;
- rechaza entradas inválidas;
- mantiene resultados después de serializar;
- supera las pruebas automatizadas.
